In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf latex2sympy2 sympy
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [ ]:
import os, sys, re, time, torch
import sympy as sp
from sympy import symbols, simplify, N

BASE_DIR    = '/content/gdrive/MyDrive/Colab Notebooks/NLP_assignment'
PACKAGE_DIR = os.path.join(BASE_DIR, 'millionaire_client')

if not os.path.exists(BASE_DIR):
    print(f"Error: Path {BASE_DIR} not found. Please check your Google Drive paths.")

sys.path.append(BASE_DIR)
print("✓ Environment Ready")

✓ Environment Ready


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "Qwen/Qwen2.5-Math-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# فشرده سازی 4 بیتی حذف شد. مدل به صورت خام و بسیار سریع روی T4 لود می شود
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16
)
model.eval()
print(f"✓ Model loaded in FAST MODE: {MODEL_ID}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

✓ Model loaded in FAST MODE: Qwen/Qwen2.5-Math-1.5B-Instruct


In [ ]:
import re
import torch
from sympy import symbols, simplify, N

def solve_latex_expression(question_text: str) -> str | None:
    try:
        from latex2sympy2 import latex2sympy
        latex_blocks = re.findall(r'\$([^$]+)\$', question_text)
        if not latex_blocks: return None
        main_latex = max(latex_blocks, key=len)
        expr = latex2sympy(main_latex)
        result = float(N(simplify(expr)))
        return str(int(result)) if result == int(result) else str(round(result, 6))
    except Exception:
        return None

def match_result_to_option(computed: str, options: list) -> int | None:
    try: computed_val = float(computed)
    except ValueError: return None
    for i, opt in enumerate(options):
        opt_clean = re.sub(r'\\[a-zA-Z]+\{?|\}', '', opt.text).strip()
        for n in re.findall(r'-?\d+\.?\d*', opt_clean):
            try:
                if abs(float(n) - computed_val) < 1e-4: return i
            except ValueError: continue
    return None

def classify_math_type(text: str) -> str:
    text = text.lower()
    if '$' in text and re.search(r'\$[^$]+\$', text): return 'latex_algebra'
    if any(k in text for k in ['standard deviation', 'probability', 'z-score', 't-test', 'mean']): return 'stats_probability'
    if any(k in text for k in ['derivative', 'integral', 'velocity', 'rate']): return 'calculus'
    return 'conceptual'

def choose_answer_math_fixed(question, tokenizer, model) -> tuple:
    options_text = "\n".join(f"{chr(65+i)}) {opt.text}" for i, opt in enumerate(question.options))
    math_type = classify_math_type(question.text)

    # 1. SymPy برای حل سریع
    if math_type == 'latex_algebra':
        computed = solve_latex_expression(question.text)
        if computed:
            matched_idx = match_result_to_option(computed, question.options)
            if matched_idx is not None:
                return question.options[matched_idx].id, chr(65 + matched_idx), f"sympy→{computed}"

    # 2. پرامپت با محدودیت توکن بالاتر برای جلوگیری از قطع شدن استدلال
    messages = [
        {"role": "system", "content": "You are a math solver. Reason step-by-step but BE EXTREMELY BRIEF. DO NOT write long paragraphs. Conclude with your final option letter inside a box, like \\boxed{A}, \\boxed{B}, \\boxed{C}, or \\boxed{D}."},
        {"role": "user", "content": f"{question.text}\n\nOptions:\n{options_text}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500, # افزایش داده شد تا بتواند استدلال را تمام کند (تا 25 ثانیه جا دارد)
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    raw = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    print(f"\n[Model Log]\n{raw.strip()}\n[End Log]")

    # 3. سیستم استخراج

    # اولویت اول: پیدا کردن \boxed{X}
    match = re.search(r'\\boxed\{([A-D])\}', raw, re.IGNORECASE)
    if match:
        letter = match.group(1).upper()
        idx = ['A', 'B', 'C', 'D'].index(letter)
        return question.options[idx].id, letter, f"boxed_{math_type}"

    # اولویت دوم: پیدا کردن کلماتی مثل Option X یا Answer: X
    match = re.search(r'(?:FINAL ANSWER|option|answer is|choice)\s*[:]?\s*([A-D])\b', raw, re.IGNORECASE)
    if match:
        letter = match.group(1).upper()
        idx = ['A', 'B', 'C', 'D'].index(letter)
        return question.options[idx].id, letter, f"text_match_{math_type}"

    # اولویت سوم (فال‌بک نهایی): آخرین حرف انگلیسی مجزا در متن
    matches = re.findall(r'\b([A-D])\b', raw.strip().upper())
    letter = matches[-1] if matches else 'C'
    try:
        idx = ['A', 'B', 'C', 'D'].index(letter)
    except ValueError:
        idx = 2
        letter = 'C'

    return question.options[idx].id, letter, f"fallback_{math_type}"

In [ ]:
from millionaire_client import MillionaireClient
from millionaire_client.exceptions import TimeoutError, RateLimitError

client = MillionaireClient('http://131.175.15.22:51111/')
user   = client.login('SYNC', 'SYNCSYNC')
print(f"✓ Logged in as: {user.username}")

competitions = client.competitions.list_all()
# فرض بر این است که ایندکس 3 مربوط به Maths است. اگر تغییر کرده، عدد 3 را اصلاح کن
COMPETITION_ID = competitions[3].id

game = client.game.start(competition_id=COMPETITION_ID, mode='text')

while game.in_progress:
    question = game.current_question
    if question is None: break

    print(f"\n{'='*60}\nLevel: {game.current_level}\nQ: {question.text}")
    for i, opt in enumerate(question.options):
        print(f"  {chr(65+i)}) {opt.text}")

    t0 = time.time()
    option_id, letter, method = choose_answer_math_fixed(question, tokenizer, model)
    t1 = time.time()

    print(f"→ Predicted: {letter} | Method: {method} | Time taken: {t1-t0:.2f}s")

    try:
        result = game.answer(option_id)
        print(f"Correct: {result.correct} | Earned: {result.earned_amount}")
    except TimeoutError:
        print("✗ Timed out (Generation took >30s)")
        break
    except RateLimitError:
        print("⚠ Rate limited — waiting 5s")
        time.sleep(5)
        result = game.answer(option_id)
        print(f"Correct: {result.correct} | Earned: {result.earned_amount}")

    if result.game_over: break
    time.sleep(1)

print(f"\n{'='*60}\n✓ Game over. Final score: {game.earned_amount}")

✓ Logged in as: SYNC

Level: 1
Q: Find $x$, given that\[\dfrac{\sqrt{x}}{x\sqrt{3}+\sqrt{2}} = \dfrac{1}{2x\sqrt{6}+4}.\]
  A) \frac{1}{4}
  B) \frac{3}{4}
  C) \frac{1}{2}
  D) \frac{1}{8}
